# RecycleVision - GPU training on Google Colab
Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.

Steps: get the code -> download the 3 Kaggle datasets -> rebuild the exact de-duplicated dataset from `data/manifest.csv` -> train 4 models -> evaluate -> zip results. Trained models are written straight to Google Drive, so after a disconnect just Run all again: finished models are skipped.

**Note:** the repository already contains the trained models and results from the original run. Running this notebook retrains everything from scratch (about 1.5 h on a free T4). Afterwards, run `scripts/evaluate_models.py` once more on a CPU machine to measure CPU latency (Colab measures GPU).

In [ ]:
!nvidia-smi -L
import tensorflow as tf, keras, sys
print(sys.version.split()[0], 'TF', tf.__version__, 'Keras', keras.__version__, tf.config.list_physical_devices('GPU'))

## 1. Code and manifest

In [ ]:
!git clone -q https://github.com/aryaman2213/RecycleVision-Garbage-Image-Classification-Using-Deep-Learning.git /content/recyclevision 2>/dev/null || git -C /content/recyclevision pull -q
!ls /content/recyclevision

## 2. Datasets (public Kaggle downloads, size-checked against the manifest's versions)

In [ ]:
!mkdir -p /content/raw
!curl -sSL -o /content/raw/garbage-classification-12.zip https://www.kaggle.com/api/v1/datasets/download/mostafaabla/garbage-classification
!curl -sSL -o /content/raw/garbage-classification-v2.zip https://www.kaggle.com/api/v1/datasets/download/sumn2u/garbage-classification-v2
!curl -sSL -o /content/raw/garbage-classification-6.zip https://www.kaggle.com/api/v1/datasets/download/asdasdasasdas/garbage-classification
!ls -l /content/raw

In [ ]:
%cd /content/recyclevision
!RECYCLEVISION_RAW=/content/raw python scripts/build_dataset.py

## 3. Persist outputs to Google Drive (approve the Drive pop-up)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/recyclevision_outputs'
!mkdir -p {OUT}/models {OUT}/reports && rm -rf models reports && ln -s {OUT}/models models && ln -s {OUT}/reports reports
!ls -l models/ reports/

## 4. Train (one cell per model, so progress is visible and resumable)

In [ ]:
!python scripts/train_models.py --models mobilenetv2

In [ ]:
!python scripts/train_models.py --models efficientnetb0

In [ ]:
!python scripts/train_models.py --models resnet50

In [ ]:
!python scripts/train_models.py --models baseline_cnn

## 5. Evaluate on the held-out test set

In [ ]:
!python scripts/evaluate_models.py

## 6. Download everything

In [ ]:
!cd /content && rm -f recyclevision_results.zip && zip -qr recyclevision_results.zip recyclevision/models/ recyclevision/reports/ && ls -lh recyclevision_results.zip
from google.colab import files
files.download('/content/recyclevision_results.zip')